# Metasyn additional features tutorial

In this tutorial we will explore features that go beyond the standard fit-and-synthesise loop. We will cover how to make columns depend on each other, how to apply different distributions based on the value of another column, and how to derive boolean columns from numeric or categorical constraints.


#TODO:
- write down all logical expressions and conditions (implemented dunders)
- create a better example for adults. Because the whole table should be adjusted if we include children in the csv.
- or make sure the syntesized data complies with childrens height and weight.

### Install and import

First, let's install metasyn if you haven't done so already.

In [11]:
# %pip install metasyn

Now let's import the packages we need.

In [12]:
import polars as pl

from metasyn import demo_data
from metasyn.builder import MetaFrameBuilder
from metasyn.distribution import DiscreteTruncatedNormalDistribution
from metasyn.distribution.base import (
    ColumnReference,
    GreaterThanCondition,
    IfThenElse,
)

### Synthesising time-series data

A common challenge with time-series datasets is that columns can be related to each other. For example, in a hospital admissions dataset the `discharge_date` should always come *after* the `admission_date`. By default, metasyn treats each column independently, so this constraint is not preserved.

Let's load our hospital admissions dataset to see this in action.

In [24]:
df = pl.read_csv("../metasyn/demo/demo_hospital_2.csv")
df = df.with_columns(
    pl.col("patient_id").cast(pl.Int64),
    pl.col("admission_date").str.to_datetime(),
    pl.col("discharge_date").str.to_datetime(),
    pl.col("length_of_stay").str.extract(r"(\d+)").cast(pl.Int64),
    pl.col("sex").cast(pl.Categorical),
)

df

patient_id,admission_date,discharge_date,length_of_stay,age,height_cm,weight_kg,sex
i64,datetime[μs],datetime[μs],i64,i64,i64,i64,cat
1,2023-01-04 00:00:00,2023-01-06 00:00:00,2,44,164,67,"""F"""
2,2023-01-08 00:00:00,2023-01-18 00:00:00,10,78,154,57,"""F"""
3,2023-01-08 00:00:00,2023-01-19 00:00:00,11,18,167,77,"""M"""
4,2023-01-08 00:00:00,2023-01-24 00:00:00,16,83,177,95,"""F"""
5,2023-01-15 00:00:00,2023-01-21 00:00:00,6,9,138,25,"""M"""
…,…,…,…,…,…,…,…
96,2024-12-10 00:00:00,2024-12-21 00:00:00,11,81,173,71,"""M"""
97,2024-12-19 00:00:00,2024-12-25 00:00:00,6,21,180,70,"""M"""
98,2024-12-19 00:00:00,2025-01-04 00:00:00,16,9,152,30,"""M"""


If we fit and synthesise without any additional instructions, the dates are generated independently. Let's check how often this produces invalid results.

In [14]:
builder = MetaFrameBuilder()
builder.add_dataframe(df, None)

synth_unconstrained = builder.fit().synthesize()

n_invalid = (synth_unconstrained["discharge_date"] < synth_unconstrained["admission_date"]).sum()
print(f"\nRows where discharge_date < admission_date: {n_invalid} / {len(synth_unconstrained)}")

synth_unconstrained.head()

  patient_id: 100%|██████████| 8/8 [00:00<00:00, 760.49variables/s]


Rows where discharge_date < admission_date: 43 / 100


patient_id,admission_date,discharge_date,length_of_stay,age,height_cm,weight_kg,sex
i64,datetime[μs],datetime[μs],i64,i64,i64,i64,cat
1,2023-04-11 00:00:00,2023-02-06 00:00:00,13,83,126,47,"""M"""
2,2023-09-06 00:00:00,2023-07-20 00:00:00,6,18,108,72,"""F"""
3,2023-04-16 00:00:00,2023-09-19 00:00:00,8,68,160,106,"""M"""
4,2024-04-28 00:00:00,2024-06-01 00:00:00,2,37,138,66,"""M"""
5,2024-05-14 00:00:00,2025-01-07 00:00:00,8,70,170,53,"""M"""


As we can see, roughly half the rows are invalid. To fix this, we can add a hidden `duration_days` column that holds the difference in days between `discharge_date` and `admission_date`. Metasyn fits a distribution to the new column, which can then be used to compute `discharge_date = admission_date + duration_days`. Setting `hidden=True` ensures the helper column does **not** appear in the final output.

Note that `ColumnReference` refers to the values in a column at synthesis time, and can be used to build expressions between columns.

In [23]:
builder = MetaFrameBuilder()
builder.add_dataframe(df, None)

# Add a column and mark as hidden
builder.add_column("duration_days", hidden=True)

# Create a series for the difference between discharge_date and admission_date, and use it to derive discharge_date
builder["duration_days"].series = ColumnReference("discharge_date") - ColumnReference("admission_date")
builder["discharge_date"].distribution = ColumnReference("admission_date") + ColumnReference("duration_days")

synth = builder.fit().synthesize()

n_invalid = (synth["discharge_date"] <= synth["admission_date"]).sum()
print(f"\nRows where discharge_date ≤ admission_date: {n_invalid} / {len(synth)}")

synth

  patient_id: 100%|██████████| 10/10 [00:00<00:00, 783.02variables/s]


Rows where discharge_date ≤ admission_date: 0 / 100


patient_id,admission_date,discharge_date,length_of_stay,age,height_cm,weight_kg,sex,Adult
i64,datetime[μs],datetime[μs],i64,i64,i64,i64,cat,bool
1,2024-08-17 00:00:00,2024-08-26 00:00:00,4,61,174,75,"""M""",true
2,2024-02-13 00:00:00,2024-02-17 00:00:00,19,90,150,69,"""F""",false
3,2024-12-10 00:00:00,2024-12-21 00:00:00,15,88,164,76,"""M""",true
4,2023-10-15 00:00:00,2023-10-19 00:00:00,6,50,161,68,"""F""",false
5,2024-06-18 00:00:00,2024-07-03 00:00:00,11,34,187,94,"""F""",true
…,…,…,…,…,…,…,…,…
96,2024-05-04 00:00:00,2024-05-10 00:00:00,13,47,169,51,"""F""",true
97,2024-01-25 00:00:00,2024-02-13 00:00:00,6,87,160,86,"""M""",true
98,2024-08-21 00:00:00,2024-08-22 00:00:00,3,13,199,78,"""M""",true


No more invalid rows.

### Logical expressions and conditions

When a dataset has a boolean column that is fully determined by another column (e.g. `Grownup = Age > 18`), we can encode that constraint explicitly using `GreaterThanCondition`. This ensures the synthesised data is always internally consistent without any post-processing.


The comparison and logical operators let you derive boolean columns from expressions.
They can be combined with `&` (AND) and `|` (OR) to form compound conditions.

| Condition class | Operator | Example |
|---|---|---|
| `GreaterThanCondition(a, b)` | `a > b` | `Age > 18` |
| `LessThanCondition(a, b)` | `a < b` | `Age < 3` |
| `EqualsCondition(a, b)` | `a == b` | `Sex == "male"` |
| `NotEqualsCondition(a, b)` | `a != b` | `Embarked != "S"` |
| `AndOperator(a, b)` | `a & b` | `Adult & (Sex == "male")` |
| `OrOperator(a, b)` | `a \| b` | `Adult \| Child` |

The table below shows these in action, using the Titanic dataset.

In [22]:
from metasyn.distribution.base import LessThanCondition

# Add derived boolean columns to Titanic
df = df.with_columns(
    (pl.col("age") > 18).alias("Adult"),
)

builder_cond = MetaFrameBuilder()
builder_cond.add_dataframe(df, None)
builder_cond["Adult"].distribution = ColumnReference("age") > 18


synth_cond = builder_cond.fit().synthesize()
synth_cond

  patient_id: 100%|██████████| 9/9 [00:00<00:00, 912.64variables/s]


patient_id,admission_date,discharge_date,length_of_stay,age,height_cm,weight_kg,sex,Adult
i64,datetime[μs],datetime[μs],i64,i64,i64,i64,cat,bool
1,2024-09-21 00:00:00,2023-01-18 00:00:00,2,30,199,70,"""M""",true
2,2023-09-27 00:00:00,2024-12-16 00:00:00,13,38,190,57,"""F""",true
3,2023-02-22 00:00:00,2023-06-04 00:00:00,15,47,125,66,"""M""",true
4,2024-04-13 00:00:00,2023-04-15 00:00:00,4,89,146,76,"""M""",true
5,2024-02-05 00:00:00,2023-05-04 00:00:00,11,52,160,65,"""M""",true
…,…,…,…,…,…,…,…,…
96,2023-01-20 00:00:00,2024-05-20 00:00:00,10,87,180,67,"""M""",true
97,2023-12-02 00:00:00,2024-11-18 00:00:00,18,57,167,72,"""F""",true
98,2023-08-24 00:00:00,2023-10-04 00:00:00,2,50,160,61,"""M""",true


#### IfThenElse example

Sometimes the distribution of a column depends on the value of another column. For example, `height_cm` tends to differ between male and female patients. With `IfThenElse`, you can specify a different distribution or value for each group.

In [ ]:
builder_if = MetaFrameBuilder()
builder_if.add_dataframe(df, None)

# Males: TruncatedNormal centred at 180 cm; females: centred at 170 cm
builder_if["height_cm"].distribution = IfThenElse(
    ColumnReference("sex") == "M",
    DiscreteTruncatedNormalDistribution(lower=160, upper=200, mean=180, sd=10),
    DiscreteTruncatedNormalDistribution(lower=150, upper=190, mean=170, sd=10),
)

synth_if = builder_if.fit().synthesize()
synth_if[["sex", "height_cm"]].head(10)

  patient_id: 100%|██████████| 9/9 [00:00<00:00, 550.25variables/s]


sex,height_cm
cat,i64
"""F""",167
"""F""",168
"""F""",161
"""M""",198
"""F""",158
"""M""",181
"""F""",176
"""M""",199
"""M""",171


In [ ]:
# # Add a derived boolean column to the Titanic dataframe
# df_titanic_ext = df_titanic.with_columns(
#     (pl.col("Age") > 18).alias("Grownup")
# )

# builder_g = MetaFrameBuilder()
# builder_g.add_dataframe(df_titanic_ext, None)

# # Grownup must always equal Age > 18
# builder_g["Grownup"].distribution = GreaterThanCondition(ColumnReference("Age"), 18)

# synth_g = builder_g.fit().synthesize()


NameError: name 'df_titanic' is not defined